<a href="https://colab.research.google.com/github/cintiajuarez-arch/proyecto_mineria_datos_2_2da_entrega/blob/master/Proyecto2_CintiaJuarez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark -q

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CloudProviderAnalytics") \
    .getOrCreate()

# Verificamos que Spark esté activo mostrando su versión
print(f"SparkSession iniciada con éxito. Versión de Spark: {spark.version}")

SparkSession iniciada con éxito. Versión de Spark: 4.0.5


In [4]:
# Inspeccion de los archivos CSV (BATCH)
# Ruta base
LANDING_PATH = "/content/drive/MyDrive/Minería de Datos II/Proyectos/datalake/landing"

# Leemos el CSV de Organizaciones / Clientes
customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{LANDING_PATH}/customers_orgs.csv")

print("=== ESQUEMA: customers_orgs.csv ===")
customers_df.printSchema()

print("\n=== PRIMEROS 5 REGISTROS ===")
customers_df.show(5, truncate=False)

print(f"Total de registros en clientes: {customers_df.count()}")

# Leemos el CSV de Facturación Mensual
billing_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{LANDING_PATH}/billing_monthly.csv")

print("\n=== ESQUEMA: billing_monthly.csv ===")
billing_df.printSchema()

print("\n=== PRIMEROS 5 REGISTROS ===")
billing_df.show(5, truncate=False)

=== ESQUEMA: customers_orgs.csv ===
root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)


=== PRIMEROS 5 REGISTROS ===
+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_id      |org_name       |industry     |hq_region |plan_tier|is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|
+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_xaji0y6d|Nimbus Labs 0  

Inspección de los eventos .jsonl (Streaming)

In [5]:
STREAM_PATH = f"{LANDING_PATH}/usage_events_stream/*.jsonl"

# Leemos todos los archivos JSONL de la carpeta
events_df = spark.read \
    .json(STREAM_PATH)

print("=== ESQUEMA GENERAL DE LOS EVENTOS DE USO (JSONL) ===")
events_df.printSchema()

print("\n=== MUESTRA DE EVENTOS DE USO ===")
events_df.show(5, truncate=False)

=== ESQUEMA GENERAL DE LOS EVENTOS DE USO (JSONL) ===
root
 |-- carbon_kg: double (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- event_id: string (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- metric: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- schema_version: long (nullable = true)
 |-- service: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: string (nullable = true)


=== MUESTRA DE EVENTOS DE USO ===
+---------+------------------+----------------+------------+----------------+------------+----------+------------+--------------+---------+--------------------+--------+-------+
|carbon_kg|cost_usd_increment|event_id        |genai_tokens|metric          |org_id      |region    |resource_id |schema_version|service  |timestamp           |unit    |value  |
+---------+-----------

Capa Bronze (Batch)

In [6]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    BooleanType, DateType, LongType, TimestampType
)
from pyspark.sql.functions import current_timestamp, input_file_name

# 1. Configuración de rutas principales en tu Google Drive
BASE_PATH = "/content/drive/MyDrive/Minería de Datos II/Proyectos/datalake"
LANDING_PATH = f"{BASE_PATH}/landing"
BRONZE_PATH = f"{BASE_PATH}/bronze"

print("Iniciando la ingesta a la Capa Bronze...")

# --- 2. DEFINICIÓN DE ESQUEMAS EXPLÍCITOS (STRUCTTYPE) ---

# Schema: customers_orgs
customers_schema = StructType([
    StructField("org_id", StringType(), True),
    StructField("org_name", StringType(), True),
    StructField("industry", StringType(), True),
    StructField("hq_region", StringType(), True),
    StructField("plan_tier", StringType(), True),
    StructField("is_enterprise", BooleanType(), True),
    StructField("signup_date", DateType(), True),
    StructField("sales_rep", StringType(), True),
    StructField("lifecycle_stage", StringType(), True),
    StructField("marketing_source", StringType(), True),
    StructField("nps_score", DoubleType(), True)
])

# Schema: billing_monthly
billing_schema = StructType([
    StructField("invoice_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("month", DateType(), True),
    StructField("subtotal", DoubleType(), True),
    StructField("credits", DoubleType(), True),
    StructField("taxes", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("exchange_rate_to_usd", DoubleType(), True)
])

# Schema: users
users_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("role", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("is_active", BooleanType(), True)
])

# Schema: resources
resources_schema = StructType([
    StructField("resource_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("status", StringType(), True)
])

# Schema: support_tickets
tickets_schema = StructType([
    StructField("ticket_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("opened_at", TimestampType(), True),
    StructField("closed_at", TimestampType(), True),
    StructField("severity", StringType(), True),
    StructField("category", StringType(), True),
    StructField("sla_breached", BooleanType(), True),
    StructField("csat", DoubleType(), True)
])

# Schema: marketing_touches
marketing_schema = StructType([
    StructField("touch_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("touch_date", DateType(), True),
    StructField("converted", BooleanType(), True)
])

# Schema: nps_surveys
nps_schema = StructType([
    StructField("survey_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("survey_date", DateType(), True),
    StructField("score", DoubleType(), True)
])


# --- 3. PROCESAMIENTO E INGESTA BATCH A BRONZE (PARQUET) ---

datasets = [
    ("customers_orgs.csv", customers_schema, "customers_orgs"),
    ("billing_monthly.csv", billing_schema, "billing_monthly"),
    ("users.csv", users_schema, "users"),
    ("resources.csv", resources_schema, "resources"),
    ("support_tickets.csv", tickets_schema, "support_tickets"),
    ("marketing_touches.csv", marketing_schema, "marketing_touches"),
    ("nps_surveys.csv", nps_schema, "nps_surveys")
]

for file_name, schema, table_name in datasets:
    print(f"Procesando {file_name} -> Bronze ({table_name})...")

    df = spark.read \
        .option("header", "true") \
        .schema(schema) \
        .csv(f"{LANDING_PATH}/{file_name}") \
        .withColumn("ingest_ts", current_timestamp()) \
        .withColumn("source_file", input_file_name())

    # Escribimos en formato Parquet
    df.write \
        .mode("overwrite") \
        .parquet(f"{BRONZE_PATH}/{table_name}")

print("\n¡Todos los maestros se ingestaron con éxito en la Capa Bronze!")

# --- 4. VERIFICACIÓN DE LECTURA DESDE PARQUET ---

print("\n--- Verificación de la tabla Bronze 'billing_monthly' ---")
df_check = spark.read.parquet(f"{BRONZE_PATH}/billing_monthly")
df_check.printSchema()
df_check.show(3, truncate=False)

Iniciando la ingesta a la Capa Bronze...
Procesando customers_orgs.csv -> Bronze (customers_orgs)...
Procesando billing_monthly.csv -> Bronze (billing_monthly)...
Procesando users.csv -> Bronze (users)...
Procesando resources.csv -> Bronze (resources)...
Procesando support_tickets.csv -> Bronze (support_tickets)...
Procesando marketing_touches.csv -> Bronze (marketing_touches)...
Procesando nps_surveys.csv -> Bronze (nps_surveys)...

¡Todos los maestros se ingestaron con éxito en la Capa Bronze!

--- Verificación de la tabla Bronze 'billing_monthly' ---
root
 |-- invoice_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- month: date (nullable = true)
 |-- subtotal: double (nullable = true)
 |-- credits: double (nullable = true)
 |-- taxes: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- exchange_rate_to_usd: double (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

+------------+----------

Capa Bronze (Streaming / JSONL)

In [7]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType
)
from pyspark.sql.functions import current_timestamp, input_file_name, to_timestamp

# 1. Esquema explícito unificado para los eventos JSONL (Soporta v1 y v2)
events_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("resource_id", StringType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("metric", StringType(), True),
    StructField("value", StringType(), True),             # Llega como String en Landing
    StructField("unit", StringType(), True),
    StructField("cost_usd_increment", DoubleType(), True),
    StructField("schema_version", LongType(), True),
    StructField("genai_tokens", LongType(), True),       # Presente desde v2
    StructField("carbon_kg", DoubleType(), True)          # Presente desde v2
])

CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/usage_events"

print("Iniciando Structured Streaming de usage_events_stream (*.jsonl) -> Bronze...")

# 2. Lectura en modo STREAMING desde el directorio de Landing
events_stream_df = spark.readStream \
    .schema(events_schema) \
    .json(f"{LANDING_PATH}/usage_events_stream") \
    .withColumn("ingest_ts", current_timestamp()) \
    .withColumn("source_file", input_file_name())

# 3. Escritura en STREAMING hacia Bronze con Checkpoint (Garantiza Idempotencia y Tolerancia a Fallos)
query = events_stream_df.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .option("path", f"{BRONZE_PATH}/usage_events") \
    .start()

# Dejamos correr el stream por unos segundos para procesar los micro-lotes existentes y luego detenemos
import time
time.sleep(10)
query.stop()

print("¡Micro-lotes procesados e ingestados con éxito en Bronze mediante Structured Streaming!")

# 4. Verificación de la lectura en Parquet
df_events_check = spark.read.parquet(f"{BRONZE_PATH}/usage_events")
print("\n--- Verificación de la tabla Bronze 'usage_events' ---")
df_events_check.printSchema()
print(f"Total de eventos ingestados en Bronze: {df_events_check.count()}")
df_events_check.show(3, truncate=False)

Iniciando Structured Streaming de usage_events_stream (*.jsonl) -> Bronze...
¡Micro-lotes procesados e ingestados con éxito en Bronze mediante Structured Streaming!

--- Verificación de la tabla Bronze 'usage_events' ---
root
 |-- event_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- region: string (nullable = true)
 |-- metric: string (nullable = true)
 |-- value: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- schema_version: long (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- carbon_kg: double (nullable = true)
 |-- ingest_ts: timestamp (nullable = false)
 |-- source_file: string (nullable = false)

Total de eventos ingestados en Bronze: 43200
+----------------+--------------------+------------+------------+---------+----------+----------------+-------+-

Perfilado Cuantitativo de Datos

In [8]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

def perfilar_tabla(df, nombre_tabla):
    total_filas = df.count()
    # Contar filas completamente duplicadas
    duplicados = total_filas - df.dropDuplicates().count()

    print(f"\n==================================================")
    print(f" PERFILADO CUANTITATIVO: {nombre_tabla.upper()}")
    print(f" Total Registros: {total_filas} | Filas Duplicadas: {duplicados}")
    print(f"==================================================")

    resumen_cols = []
    # Diccionario de tipos de datos de las columnas
    tipos_dict = dict(df.dtypes)

    for field in df.schema.fields:
        col_name = field.name
        col_type = field.dataType

        # Si la columna es de tipo String, evaluamos nulos o texto vacío
        if isinstance(col_type, StringType):
            nulos = df.filter(F.col(col_name).isNull() | (F.col(col_name) == "")).count()
        else:
            # Para Boolean, Double, Timestamp, etc., solo verificamos isNull()
            nulos = df.filter(F.col(col_name).isNull()).count()

        pct_nulos = round((nulos / total_filas) * 100, 2) if total_filas > 0 else 0
        cardinalidad = df.select(col_name).distinct().count()

        resumen_cols.append({
            "Columna": col_name,
            "Tipo": tipos_dict[col_name],
            "Nulos": nulos,
            "% Nulos": f"{pct_nulos}%",
            "Cardinalidad": cardinalidad
        })

    df_resumen = spark.createDataFrame(resumen_cols)
    df_resumen.show(len(df.columns), truncate=False)

# Ejecutamos el perfilado nuevamente
df_customers_br = spark.read.parquet(f"{BRONZE_PATH}/customers_orgs")
df_billing_br = spark.read.parquet(f"{BRONZE_PATH}/billing_monthly")
df_events_br = spark.read.parquet(f"{BRONZE_PATH}/usage_events")

perfilar_tabla(df_customers_br, "customers_orgs (Bronze)")
perfilar_tabla(df_billing_br, "billing_monthly (Bronze)")
perfilar_tabla(df_events_br, "usage_events (Bronze)")


 PERFILADO CUANTITATIVO: CUSTOMERS_ORGS (BRONZE)
 Total Registros: 80 | Filas Duplicadas: 0
+-------+------------+----------------+-----+---------+
|% Nulos|Cardinalidad|Columna         |Nulos|Tipo     |
+-------+------------+----------------+-----+---------+
|0.0%   |80          |org_id          |0    |string   |
|0.0%   |80          |org_name        |0    |string   |
|0.0%   |10          |industry        |0    |string   |
|0.0%   |7           |hq_region       |0    |string   |
|0.0%   |4           |plan_tier       |0    |string   |
|0.0%   |2           |is_enterprise   |0    |boolean  |
|0.0%   |40          |signup_date     |0    |date     |
|0.0%   |5           |sales_rep       |0    |string   |
|0.0%   |5           |lifecycle_stage |0    |string   |
|0.0%   |5           |marketing_source|0    |string   |
|13.75% |46          |nps_score       |11   |double   |
|0.0%   |1           |ingest_ts       |0    |timestamp|
|0.0%   |1           |source_file     |0    |string   |
+-------+--

Capa Silver y Cuarentena de Eventos

In [9]:
from pyspark.sql import functions as F

# 1. Definir rutas de Silver y Quarantine
SILVER_PATH = f"{BASE_PATH}/silver"
QUARANTINE_PATH = f"{BASE_PATH}/silver/quarantine"

print("Iniciando procesamiento de Capa Silver para 'usage_events'...")

# 2. Leer eventos desde Bronze Parquet
df_events_br = spark.read.parquet(f"{BRONZE_PATH}/usage_events")

# 3. Castear campos numéricos/temporales y estandarizar tipos
df_events_typed = df_events_br \
    .withColumn("value_num", F.col("value").cast("double")) \
    .withColumn("event_timestamp", F.to_timestamp(F.col("timestamp")))

# 4. Evaluación de Reglas de Calidad
# Regla 1: event_id no nulo
cond_r1 = F.col("event_id").isNotNull()

# Regla 2: cost_usd_increment no negativo anómalo (>= -0.01)
cond_r2 = F.col("cost_usd_increment").isNull() | (F.col("cost_usd_increment") >= -0.01)

# Regla 3: Si existe value_num, unit no debe ser nulo
cond_r3 = F.when(F.col("value_num").isNotNull(), F.col("unit").isNotNull()).otherwise(True)

# Crear columna indicadora de validez y motivo de falla
df_events_evaluated = df_events_typed.withColumn(
    "is_valid", cond_r1 & cond_r2 & cond_r3
).withColumn(
    "quarantine_reason",
    F.when(~cond_r1, "REGLA_1_EVENT_ID_NULL")
     .when(~cond_r2, "REGLA_2_COSTO_NEGATIVO_ANOMALO")
     .when(~cond_r3, "REGLA_3_UNIT_NULL_CON_VALUE")
     .otherwise("VALIDO")
)

# 5. Separación de datos VÁLIDOS y CUARENTENA
df_silver_events = df_events_evaluated.filter(F.col("is_valid") == True) \
    .drop("is_valid", "quarantine_reason") \
    .withColumn("processed_ts", F.current_timestamp())

df_quarantine_events = df_events_evaluated.filter(F.col("is_valid") == False) \
    .withColumn("quarantined_ts", F.current_timestamp())

# 6. Escritura en Parquet
# Escribimos los válidos a Silver
df_silver_events.write \
    .mode("overwrite") \
    .parquet(f"{SILVER_PATH}/usage_events")

# Escribimos los inválidos a Quarantine
df_quarantine_events.write \
    .mode("overwrite") \
    .parquet(f"{QUARANTINE_PATH}/usage_events_quarantine")

# 7. Imprimir métricas de control
total_bronze = df_events_br.count()
total_silver = df_silver_events.count()
total_quarantine = df_quarantine_events.count()

print(f"\n==================================================")
print(f" RESUMEN DE PROCESAMIENTO A SILVER")
print(f" Total en Bronze:      {total_bronze}")
print(f" Promovidos a Silver:  {total_silver} ({round(total_silver/total_bronze*100, 2)}%)")
print(f" Enviados a Cuarentena:{total_quarantine} ({round(total_quarantine/total_bronze*100, 2)}%)")
print(f"==================================================")

if total_quarantine > 0:
    print("\n--- MUESTRA DE REGISTROS EN CUARENTENA ---")
    df_quarantine_events.select("event_id", "cost_usd_increment", "value", "unit", "quarantine_reason").show(5, truncate=False)

Iniciando procesamiento de Capa Silver para 'usage_events'...

 RESUMEN DE PROCESAMIENTO A SILVER
 Total en Bronze:      43200
 Promovidos a Silver:  40956 (94.81%)
 Enviados a Cuarentena:2244 (5.19%)

--- MUESTRA DE REGISTROS EN CUARENTENA ---
+----------------+------------------+------+----+---------------------------+
|event_id        |cost_usd_increment|value |unit|quarantine_reason          |
+----------------+------------------+------+----+---------------------------+
|evt_l5ws8s604cko|10.6258           |104.0 |NULL|REGLA_3_UNIT_NULL_CON_VALUE|
|evt_qefaluuwye6h|0.085             |2.3497|NULL|REGLA_3_UNIT_NULL_CON_VALUE|
|evt_hxzquhn8l64f|0.2967            |3.5396|NULL|REGLA_3_UNIT_NULL_CON_VALUE|
|evt_o3pwrzeqscfr|10.5409           |121.0 |NULL|REGLA_3_UNIT_NULL_CON_VALUE|
|evt_r4zct0tz8yd3|0.062             |2.4644|NULL|REGLA_3_UNIT_NULL_CON_VALUE|
+----------------+------------------+------+----+---------------------------+
only showing top 5 rows


Joins y Feature Engineering (Silver Enriquecida)

In [10]:
from pyspark.sql import functions as F

print("Iniciando Joins de Enriquecimiento y Feature Engineering en Silver...")

# 1. Leer Eventos Silver y Maestro de Clientes Bronze/Silver
df_silver_events = spark.read.parquet(f"{SILVER_PATH}/usage_events")
df_customers = spark.read.parquet(f"{BRONZE_PATH}/customers_orgs")

# 2. Join de Enriquecimiento (Left Join para conservar la totalidad de los eventos)
df_events_enriched = df_silver_events.join(
    df_customers.select("org_id", "org_name", "plan_tier", "industry", "hq_region"),
    on="org_id",
    how="left"
)

# 3. Feature Engineering: Extracción y cálculo de métricas derivadas
df_silver_features = df_events_enriched \
    .withColumn("event_date", F.to_date(F.col("event_timestamp"))) \
    .withColumn("daily_cost_usd", F.coalesce(F.col("cost_usd_increment"), F.lit(0.0))) \
    .withColumn("requests_count", F.when(F.col("metric") == "requests", F.col("value_num")).otherwise(0)) \
    .withColumn("genai_tokens_cnt", F.coalesce(F.col("genai_tokens"), F.lit(0))) \
    .withColumn("carbon_kg_val", F.coalesce(F.col("carbon_kg"), F.lit(0.0)))

# 4. Guardar la tabla conformada y enriquecida en Silver Parquet
df_silver_features.write \
    .mode("overwrite") \
    .parquet(f"{SILVER_PATH}/usage_events_enriched")

print("\n¡Joins y Feature Engineering completados con éxito!")
print("\n--- MUESTRA DE EVENTOS ENRIQUECIDOS Y FEATURES CALCULADOS ---")
df_silver_features.select(
    "event_id", "org_id", "org_name", "service", "event_date",
    "daily_cost_usd", "requests_count", "genai_tokens_cnt", "carbon_kg_val"
).show(5, truncate=False)

Iniciando Joins de Enriquecimiento y Feature Engineering en Silver...

¡Joins y Feature Engineering completados con éxito!

--- MUESTRA DE EVENTOS ENRIQUECIDOS Y FEATURES CALCULADOS ---
+----------------+------------+-------------+---------+----------+--------------+--------------+----------------+-------------+
|event_id        |org_id      |org_name     |service  |event_date|daily_cost_usd|requests_count|genai_tokens_cnt|carbon_kg_val|
+----------------+------------+-------------+---------+----------+--------------+--------------+----------------+-------------+
|evt_c5rhcrkjh59e|org_axeqbnhl|Nova Labs 63 |analytics|2025-08-01|1.1239        |0.0           |0               |0.002987     |
|evt_hrqcdkf8icwj|org_t1ghr09s|Beta Cloud 78|compute  |2025-07-12|1.2313        |0.0           |0               |0.0          |
|evt_3gc4zre0xgmt|org_1swjckjl|Beta Tech 57 |compute  |2025-08-18|9.7306        |110.0         |0               |0.022        |
|evt_v2ntznddiaw3|org_1t2tala7|Gamma Data 15|c

Capa Gold (Parquet)

In [11]:
from pyspark.sql import functions as F

# 1. Definir la ruta de la Capa Gold
GOLD_PATH = f"{BASE_PATH}/gold"

print("Iniciando la agregación y construcción de la Capa Gold (Mart FinOps)...")

# 2. Leer los datos enriquecidos de Silver
df_silver_enriched = spark.read.parquet(f"{SILVER_PATH}/usage_events_enriched")

# 3. Construir el Data Mart 'org_daily_usage_by_service' agrupando por (org_id, event_date, service)
df_gold_mart = df_silver_enriched \
    .groupBy(
        F.col("org_id"),
        F.col("event_date").alias("usage_date"),
        F.col("service"),
        F.col("org_name")
    ) \
    .agg(
        F.round(F.sum("daily_cost_usd"), 4).alias("daily_cost_usd"),
        F.sum("requests_count").cast("bigint").alias("total_requests"),
        F.sum("genai_tokens_cnt").cast("bigint").alias("total_genai_tokens"),
        F.round(F.sum("carbon_kg_val"), 6).alias("total_carbon_kg"),
        F.count("event_id").alias("total_events")
    ) \
    .withColumn("mart_created_ts", F.current_timestamp())

# 4. Persistir en la Capa Gold en formato Parquet
df_gold_mart.write \
    .mode("overwrite") \
    .parquet(f"{GOLD_PATH}/org_daily_usage_by_service")

# 5. Métricas de verificación
total_marts_rows = df_gold_mart.count()

print(f"\n==================================================")
print(f" RESUMEN DE LA CAPA GOLD (MART FINOPS)")
print(f" Registros agregados generados: {total_marts_rows}")
print(f" Ruta de salida: {GOLD_PATH}/org_daily_usage_by_service")
print(f"==================================================")

print("\n--- MUESTRA DEL DATA MART GOLD ('org_daily_usage_by_service') ---")
df_gold_mart.select(
    "org_id", "org_name", "usage_date", "service",
    "daily_cost_usd", "total_requests", "total_genai_tokens", "total_carbon_kg"
).show(5, truncate=False)

Iniciando la agregación y construcción de la Capa Gold (Mart FinOps)...

 RESUMEN DE LA CAPA GOLD (MART FINOPS)
 Registros agregados generados: 11050
 Ruta de salida: /content/drive/MyDrive/Minería de Datos II/Proyectos/datalake/gold/org_daily_usage_by_service

--- MUESTRA DEL DATA MART GOLD ('org_daily_usage_by_service') ---
+------------+--------------+----------+----------+--------------+--------------+------------------+---------------+
|org_id      |org_name      |usage_date|service   |daily_cost_usd|total_requests|total_genai_tokens|total_carbon_kg|
+------------+--------------+----------+----------+--------------+--------------+------------------+---------------+
|org_t1ghr09s|Beta Cloud 78 |2025-07-12|compute   |9.5458        |125           |0                 |0.0            |
|org_wnnhj7xv|Alpha Data 6  |2025-07-13|networking|0.0899        |0             |0                 |0.0            |
|org_x7eedtjv|Apex Cloud 34 |2025-07-21|networking|2.5231        |265           |0     

Estrategia de Modelado Query-First

In [12]:
from pyspark.sql import functions as F

print("=== CAPA SERVING: MODELADO CQL Y PRUEBA DE IDEMPOTENCIA ===")

# 1. Definición del modelo Query-First en DDL CQL (para documentación y entrega)
cql_schema = """
-- 1. Creación del Keyspace
CREATE KEYSPACE IF NOT EXISTS cloud_analytics
WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1};

-- 2. Creación de la Tabla Query-First para FinOps
CREATE TABLE IF NOT EXISTS cloud_analytics.org_daily_usage_by_service (
    org_id text,
    usage_date date,
    service text,
    org_name text,
    daily_cost_usd double,
    total_requests bigint,
    total_genai_tokens bigint,
    total_carbon_kg double,
    total_events bigint,
    mart_created_ts timestamp,
    PRIMARY KEY ((org_id), usage_date, service)
) WITH CLUSTERING ORDER BY (usage_date DESC, service ASC);
"""

print("Estructura CQL declarada para Cassandra/AstraDB:")
print(cql_schema)

# 2. Leer la capa Gold recién creada
df_gold = spark.read.parquet(f"{GOLD_PATH}/org_daily_usage_by_service")
conteo_inicial = df_gold.count()

# 3. Demostración de Idempotencia por Clave Natural (UPSERT/Deduplicación)
# Simulamos un reprocesamiento re-leyendo y sobreescribiendo en la zona de serving
df_gold_reprocessed = spark.read.parquet(f"{GOLD_PATH}/org_daily_usage_by_service")

# Aplicamos dropDuplicates sobre la Primary Key de Cassandra: (org_id, usage_date, service)
df_serving_idempotent = df_gold.union(df_gold_reprocessed) \
    .dropDuplicates(["org_id", "usage_date", "service"])

conteo_final = df_serving_idempotent.count()

print("\n==================================================")
print(" PRUEBA DE IDEMPOTENCIA EN CAPA SERVING")
print(f" Registros antes de re-ejecución: {conteo_inicial}")
print(f" Registros después del re-procesamiento: {conteo_final}")
if conteo_inicial == conteo_final:
    print(" RESULTADO: PASÓ LA PRUEBA - La re-ejecución no generó duplicados (Idempotencia garantizada).")
print("==================================================")

=== CAPA SERVING: MODELADO CQL Y PRUEBA DE IDEMPOTENCIA ===
Estructura CQL declarada para Cassandra/AstraDB:

-- 1. Creación del Keyspace
CREATE KEYSPACE IF NOT EXISTS cloud_analytics
WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1};

-- 2. Creación de la Tabla Query-First para FinOps
CREATE TABLE IF NOT EXISTS cloud_analytics.org_daily_usage_by_service (
    org_id text,
    usage_date date,
    service text,
    org_name text,
    daily_cost_usd double,
    total_requests bigint,
    total_genai_tokens bigint,
    total_carbon_kg double,
    total_events bigint,
    mart_created_ts timestamp,
    PRIMARY KEY ((org_id), usage_date, service)
) WITH CLUSTERING ORDER BY (usage_date DESC, service ASC);


 PRUEBA DE IDEMPOTENCIA EN CAPA SERVING
 Registros antes de re-ejecución: 11050
 Registros después del re-procesamiento: 11050
 RESULTADO: PASÓ LA PRUEBA - La re-ejecución no generó duplicados (Idempotencia garantizada).


Consultas CQL

In [13]:
from pyspark.sql import functions as F

print("=== CAPA SERVING: CONSULTAS ANALÍTICAS OBLIGATORIAS (CQL SIMULATION) ===")

# 1. Cargar el DataFrame de la Capa Serving/Gold
df_serving = spark.read.parquet(f"{GOLD_PATH}/org_daily_usage_by_service")

# Elegimos una organización de muestra existente en los datos
org_test = df_serving.select("org_id").first()["org_id"]

# -------------------------------------------------------------------------
# CONSULTA 1 (CQL): Consumo y Costos Diarios por Organización y Rango de Fechas
# -------------------------------------------------------------------------
cql_query_1 = f"""
-- CONSULTA 1: Costos y métricas de uso para la organización '{org_test}' entre Julio y Agosto 2025
SELECT org_id, usage_date, service, daily_cost_usd, total_requests, total_genai_tokens, total_carbon_kg
FROM cloud_analytics.org_daily_usage_by_service
WHERE org_id = '{org_test}'
  AND usage_date >= '2025-07-01'
  AND usage_date <= '2025-08-31';
"""

print("\n-------------------------------------------------------------------------")
print("1. EJECUCIÓN DE CONSULTA CQL #1")
print(cql_query_1)
print("--- RESULTADO OBTENIDO DESDE LA CAPA SERVING ---")

df_res_1 = df_serving.filter(
    (F.col("org_id") == org_test) &
    (F.col("usage_date") >= "2025-07-01") &
    (F.col("usage_date") <= "2025-08-31")
).orderBy(F.col("usage_date").desc())

df_res_1.select("org_id", "usage_date", "service", "daily_cost_usd", "total_requests", "total_carbon_kg").show(5, truncate=False)


# -------------------------------------------------------------------------
# CONSULTA 2 (CQL): Top Servicios por Costo Acumulado para una Organización
# -------------------------------------------------------------------------
cql_query_2 = f"""
-- CONSULTA 2: Servicios con mayor costo acumulado para la organización '{org_test}'
SELECT service, SUM(daily_cost_usd) AS total_service_cost, SUM(total_requests) AS sum_requests
FROM cloud_analytics.org_daily_usage_by_service
WHERE org_id = '{org_test}'
GROUP BY service;
"""

print("\n-------------------------------------------------------------------------")
print("2. EJECUCIÓN DE CONSULTA CQL #2")
print(cql_query_2)
print("--- RESULTADO OBTENIDO DESDE LA CAPA SERVING ---")

df_res_2 = df_serving.filter(F.col("org_id") == org_test) \
    .groupBy("service") \
    .agg(
        F.round(F.sum("daily_cost_usd"), 4).alias("total_service_cost"),
        F.sum("total_requests").alias("sum_requests")
    ) \
    .orderBy(F.col("total_service_cost").desc())

df_res_2.show(truncate=False)

=== CAPA SERVING: CONSULTAS ANALÍTICAS OBLIGATORIAS (CQL SIMULATION) ===

-------------------------------------------------------------------------
1. EJECUCIÓN DE CONSULTA CQL #1

-- CONSULTA 1: Costos y métricas de uso para la organización 'org_t1ghr09s' entre Julio y Agosto 2025
SELECT org_id, usage_date, service, daily_cost_usd, total_requests, total_genai_tokens, total_carbon_kg
FROM cloud_analytics.org_daily_usage_by_service
WHERE org_id = 'org_t1ghr09s'
  AND usage_date >= '2025-07-01'
  AND usage_date <= '2025-08-31';

--- RESULTADO OBTENIDO DESDE LA CAPA SERVING ---
+------------+----------+----------+--------------+--------------+---------------+
|org_id      |usage_date|service   |daily_cost_usd|total_requests|total_carbon_kg|
+------------+----------+----------+--------------+--------------+---------------+
|org_t1ghr09s|2025-08-31|compute   |29.5147       |346           |0.0692         |
|org_t1ghr09s|2025-08-30|networking|2.0535        |236           |0.049991       |
|or